# 🧠 Materi Kuliah Deep Learning
## Pertemuan 3: Convolutional Neural Networks (CNN)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hendrick02121977/Deep-Learning/blob/main/notebooks/03_Convolutional_Neural_Networks.ipynb)

---

**Tujuan Pembelajaran:**
- Memahami konsep Convolutional Neural Networks (CNN)
- Mengenal operasi konvolusi, pooling, dan flatten
- Memahami arsitektur CNN populer (LeNet, VGG, ResNet)
- Mengimplementasikan CNN untuk klasifikasi gambar (CIFAR-10)
- Melakukan visualisasi feature maps

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(f"TensorFlow version: {tf.__version__}")
# Cek GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU tersedia: {len(gpus) > 0}")
if gpus:
    print(f"GPU: {gpus[0].name}")

## 1. Mengapa CNN?

### Masalah dengan Fully Connected Network untuk Gambar

Bayangkan gambar 224×224×3 piksel:
- Jumlah input: 224 × 224 × 3 = **150,528 neuron**
- Dengan 1 hidden layer 1000 neuron: **150,528 × 1000 = 150 juta parameter!**

### Solusi CNN:
1. **Local connectivity**: Setiap neuron hanya terhubung ke area lokal gambar
2. **Parameter sharing**: Filter yang sama digunakan di seluruh gambar
3. **Translation invariance**: Mengenali pola di mana pun lokasinya

In [ ]:
# Demonstrasi Operasi Konvolusi
def convolve2d(image, kernel, padding=0):
    """Implementasi konvolusi 2D sederhana"""
    if padding > 0:
        image = np.pad(image, padding, mode='constant')
    
    h_img, w_img = image.shape
    h_ker, w_ker = kernel.shape
    
    h_out = h_img - h_ker + 1
    w_out = w_img - w_ker + 1
    
    output = np.zeros((h_out, w_out))
    
    for i in range(h_out):
        for j in range(w_out):
            output[i, j] = np.sum(image[i:i+h_ker, j:j+w_ker] * kernel)
    
    return output

# Buat gambar sederhana (angka 1)
image = np.array([
    [0, 0, 1, 0, 0],
    [0, 1, 1, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 1, 1, 1, 0]
], dtype=float)

# Filter/Kernel berbeda
kernels = {
    'Edge Detection\n(Horizontal)': np.array([[-1, -1, -1],
                                               [ 0,  0,  0],
                                               [ 1,  1,  1]]),
    'Edge Detection\n(Vertical)':   np.array([[-1,  0,  1],
                                               [-1,  0,  1],
                                               [-1,  0,  1]]),
    'Sharpen':                       np.array([[ 0, -1,  0],
                                               [-1,  5, -1],
                                               [ 0, -1,  0]]),
    'Blur':                          np.array([[1, 1, 1],
                                               [1, 1, 1],
                                               [1, 1, 1]]) / 9.0
}

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
fig.suptitle('Operasi Konvolusi dengan Berbagai Filter', fontsize=13, fontweight='bold')

# Tampilkan gambar asli
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Gambar Asli')
axes[0].axis('off')

# Terapkan setiap kernel
for idx, (name, kernel) in enumerate(kernels.items()):
    result = convolve2d(image, kernel)
    axes[idx+1].imshow(result, cmap='gray')
    axes[idx+1].set_title(name, fontsize=9)
    axes[idx+1].axis('off')

plt.tight_layout()
plt.show()

## 2. Arsitektur CNN

CNN terdiri dari beberapa jenis layer:

```
Input Image
     │
     ▼
┌─────────────┐
│  Conv Layer │  ← Mengekstrak fitur dengan filter
│  + ReLU     │
└─────────────┘
     │
     ▼
┌─────────────┐
│  Pooling    │  ← Mengurangi dimensi (Max/Average Pooling)
│  Layer      │
└─────────────┘
     │
  (repeat)
     │
     ▼
┌─────────────┐
│  Flatten    │  ← Ubah 2D ke 1D
└─────────────┘
     │
     ▼
┌─────────────┐
│  Dense      │  ← Klasifikasi
│  Layers     │
└─────────────┘
     │
     ▼
  Output (Softmax)
```

In [ ]:
# Load dataset CIFAR-10
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Nama kelas CIFAR-10
class_names = ['Pesawat', 'Mobil', 'Burung', 'Kucing', 'Rusa',
               'Anjing', 'Katak', 'Kuda', 'Kapal', 'Truk']

print(f"Training set: {X_train.shape}")
print(f"Test set    : {X_test.shape}")
print(f"Jumlah kelas: {len(class_names)}")
print(f"Ukuran gambar: {X_train.shape[1]}x{X_train.shape[2]}x{X_train.shape[3]} piksel")

# Normalisasi
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Visualisasi sampel
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Contoh Gambar dari Dataset CIFAR-10', fontsize=13, fontweight='bold')

for i in range(16):
    row, col = divmod(i, 8)
    axes[row, col].imshow(X_train[i])
    axes[row, col].set_title(class_names[y_train[i][0]], fontsize=8)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Build CNN Model
def build_cnn_model():
    model = keras.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Classifier
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ], name='CIFAR10_CNN')
    
    return model

tf.random.set_seed(42)
model_cnn = build_cnn_model()
model_cnn.summary()

In [ ]:
# Compile model
model_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Data Augmentation untuk meningkatkan generalisasi
datagen = keras.preprocessing.image.ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)
datagen.fit(X_train)

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

# Train model
print("Melatih CNN model pada CIFAR-10...")
print("(Ini mungkin butuh beberapa menit di CPU, lebih cepat di GPU)")

history_cnn = model_cnn.fit(
    datagen.flow(X_train, y_train, batch_size=64),
    steps_per_epoch=len(X_train) // 64,
    epochs=50,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Evaluasi model
test_loss, test_acc = model_cnn.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Hasil Training CNN pada CIFAR-10', fontsize=13, fontweight='bold')

axes[0].plot(history_cnn.history['loss'], label='Train')
axes[0].plot(history_cnn.history['val_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_cnn.history['accuracy'], label='Train')
axes[1].plot(history_cnn.history['val_accuracy'], label='Validation')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix
y_pred = np.argmax(model_cnn.predict(X_test, verbose=0), axis=1)
y_true = y_test.flatten()

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - CIFAR-10', fontsize=13, fontweight='bold')
plt.ylabel('Label Sebenarnya')
plt.xlabel('Prediksi Model')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Visualisasi Feature Maps
# Ambil model yang mengeluarkan aktivasi dari layer konvolusi
layer_outputs = [layer.output for layer in model_cnn.layers if 'conv' in layer.name]
activation_model = keras.Model(inputs=model_cnn.input, outputs=layer_outputs[:4])

# Pilih satu gambar test
sample_image = X_test[5:6]  # Ambil 1 gambar
sample_label = class_names[y_test[5][0]]

# Dapatkan aktivasi
activations = activation_model.predict(sample_image, verbose=0)

# Visualisasi feature maps dari layer pertama
first_layer_activation = activations[0]  # Shape: (1, 32, 32, 32)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle(f'Feature Maps - Layer Conv Pertama\n(Gambar: {sample_label})',
             fontsize=13, fontweight='bold')

# Tampilkan gambar asli di posisi pertama
for i in range(32):
    row, col = divmod(i, 8)
    axes[row, col].imshow(first_layer_activation[0, :, :, i], cmap='viridis')
    axes[row, col].axis('off')
    axes[row, col].set_title(f'Filter {i+1}', fontsize=6)

plt.tight_layout()
plt.show()

# Tampilkan gambar asli untuk referensi
plt.figure(figsize=(3, 3))
plt.imshow(sample_image[0])
plt.title(f'Gambar Asli: {sample_label}')
plt.axis('off')
plt.show()

## 3. Transfer Learning dengan CNN Pre-trained

Daripada melatih dari awal, kita bisa menggunakan model yang sudah dilatih pada dataset besar (ImageNet).

In [ ]:
# Demo Transfer Learning dengan MobileNetV2
print("Arsitektur CNN Pre-trained yang Populer:")
print("=" * 50)

models_info = [
    ('LeNet-5', 1998, '60K', '~98% MNIST'),
    ('AlexNet', 2012, '62M', '~57% ImageNet Top-1'),
    ('VGG-16', 2014, '138M', '~74% ImageNet Top-1'),
    ('ResNet-50', 2015, '25M', '~76% ImageNet Top-1'),
    ('InceptionV3', 2015, '24M', '~78% ImageNet Top-1'),
    ('MobileNetV2', 2018, '3.4M', '~72% ImageNet Top-1'),
    ('EfficientNetB7', 2019, '66M', '~84% ImageNet Top-1')
]

print(f"{'Model':<20} {'Tahun':<8} {'Parameter':<12} {'Accuracy'}")
print("-" * 55)
for name, year, params, acc in models_info:
    print(f"{name:<20} {year:<8} {params:<12} {acc}")

print("\nMemuat MobileNetV2 pre-trained...")

# Load MobileNetV2 tanpa layer klasifikasi (include_top=False)
base_model = keras.applications.MobileNetV2(
    input_shape=(32, 32, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze layer base model
base_model.trainable = False

# Tambah classifier baru
transfer_model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

transfer_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"\nParameter trainable: {sum(p.numpy().size for p in transfer_model.trainable_variables):,}")
print(f"Parameter frozen   : {sum(p.numpy().size for p in base_model.non_trainable_variables):,}")

## 4. Latihan Mandiri

1. **Latihan 1**: Tambah lebih banyak Conv layer. Apakah accuracy meningkat?

2. **Latihan 2**: Coba arsitektur ResNet dengan **skip connections**. Buat model dengan 2 blok residual.

3. **Latihan 3**: Gunakan Transfer Learning dengan MobileNetV2 dan latih model tersebut. Bandingkan dengan model dari scratch.

4. **Latihan 4**: Tambah lebih banyak augmentasi data (rotasi, flip, zoom). Apakah model lebih robust?

## Ringkasan

✅ **Konvolusi**: Operasi filter yang mengekstrak fitur lokal dari gambar

✅ **Pooling**: Mengurangi dimensi spasial (Max/Average Pooling)

✅ **Arsitektur CNN**: Conv → BN → ReLU → Pool → Flatten → Dense → Softmax

✅ **Data Augmentation**: Meningkatkan generalisasi dengan transformasi gambar

✅ **Feature Maps**: Representasi visual fitur yang dipelajari CNN

✅ **Transfer Learning**: Menggunakan model pre-trained untuk tugas baru

---

**Pertemuan Berikutnya:** Recurrent Neural Networks (RNN) dan LSTM untuk data sekuensial 📊